Preprocessing — Member 2: Encoding and Scaling
Online Shopper Purchase Likelihood Prediction System

This notebook loads the engineered train and test data, converts categorical
features into numeric columns, and scales numeric features for modelling.
The encoder and scaler are fitted on training data only.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath(".."))

from src.preprocessing.encoding_scaling import (
    CATEGORICAL_COLUMNS,
    build_preprocessor,
)

X_train = pd.read_csv("../data/processed/X_train_engineered.csv")
X_test = pd.read_csv("../data/processed/X_test_engineered.csv")

print("Train before encoding:", X_train.shape)
print("Test before encoding:", X_test.shape)

Train before encoding: (9764, 21)
Test before encoding: (2441, 21)


1. Encode categorical features and scale numeric features

Month, operating system, browser, region, traffic type, visitor type,
Weekend, and the interaction feature are one-hot encoded. Numeric features,
including SpecialDay, are standardized.

Member 3's current feature-selection result keeps all numeric features.
Unknown categories in test data are handled by the encoder.

In [2]:
# Member 3's current result is to keep all numeric features.
selected_numeric_columns = [
    column for column in X_train.columns
    if column not in CATEGORICAL_COLUMNS
]

selected_columns = CATEGORICAL_COLUMNS + selected_numeric_columns

preprocessor = build_preprocessor(selected_numeric_columns)

train_array = preprocessor.fit_transform(X_train[selected_columns])
test_array = preprocessor.transform(X_test[selected_columns])

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(train_array, columns=feature_names)
X_test_processed = pd.DataFrame(test_array, columns=feature_names)

assert X_train_processed.columns.equals(X_test_processed.columns)
assert np.isfinite(train_array).all()
assert np.isfinite(test_array).all()

print("Train after encoding:", X_train_processed.shape)
print("Test after encoding:", X_test_processed.shape)

Train after encoding: (9764, 83)
Test after encoding: (2441, 83)


2. Save the processed data

Save the transformed train and test sets with identical columns. Also save
the fitted preprocessor so future data can be transformed using the same
training-data rules.

In [3]:
from joblib import dump

X_train_processed.to_csv(
    "../data/processed/X_train_processed.csv", index=False
)
X_test_processed.to_csv(
    "../data/processed/X_test_processed.csv", index=False
)

dump(preprocessor, "../models/encoding_scaling.joblib")

print("Saved processed train and test CSVs.")
print("Saved fitted preprocessor to models/encoding_scaling.joblib.")

Saved processed train and test CSVs.
Saved fitted preprocessor to models/encoding_scaling.joblib.
